# IC-SHM 2026 Project 2 — Multi-view Semantic 3D Visualization

This notebook loads the **GPU-retriangulated** semantic point cloud (`semantic_bridge_gpu.ply`), applies post-processing filters (drop background, per-class SOR, deck plane), and visualizes the cleaned structure.

### Semantic Classes & Color Legend
- `0: background` — Gray `[128, 128, 128]`
- `1: deck` — Red `[255, 0, 0]`
- `2: stay_cable` — Cyan `[0, 255, 255]`
- `3: tower` — Green `[0, 255, 0]`
- `4: foundation` — Yellow `[255, 255, 0]`

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.reconstruction.visualizer import read_ply_file, create_interactive_3d_figure, CLASS_NAMES, CLASS_HEX_COLORS
from src.reconstruction.point_cloud_filter import filter_point_cloud, estimate_up_from_reconstruction

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## 1. Load GPU PLY and Apply Filters

Loads `semantic_bridge_gpu.ply` (pycolmap-cuda12 pipeline), then runs structure-only filtering: drop background, per-class SOR, deck plane, **deck core density** (removes coplanar but sparse red outliers in along/lateral), and stay-cable structural envelope. Vertical axis comes from camera poses (drone "up" ≈ gravity). The same filter works on `semantic_bridge_sparse.ply`.

In [2]:
PLY_PATH = '../outputs/point_clouds/semantic_bridge_gpu.ply'
COLMAP_MODEL = '../outputs/gpu_pipeline/triangulated'

xyz_raw, rgb_raw, cids_raw = read_ply_file(PLY_PATH)
print(f"Loaded {len(xyz_raw):,} raw points from '{PLY_PATH}'")

# Gravity direction from camera poses (drone shots have ~zero roll).
# This anchors the bridge-local frame's vertical axis for the cable envelope filter.
up = estimate_up_from_reconstruction(COLMAP_MODEL)
print(f"Camera-derived up vector: {np.round(up, 3)}")

xyz, rgb, cids, filter_stats = filter_point_cloud(xyz_raw, rgb_raw, cids_raw, up_hint=up)
print(f"Filtered -> {len(xyz):,} structure points (removed {filter_stats.initial - filter_stats.final:,})")
print(f"  removed by stage: {filter_stats.removed_by_stage}")

def class_distribution(class_ids):
    unique, counts = np.unique(class_ids, return_counts=True)
    return pd.DataFrame({
        'Class ID': unique,
        'Class Name': [CLASS_NAMES[c] for c in unique],
        'Point Count': counts,
        'Percentage (%)': (counts / len(class_ids) * 100).round(2),
    })

df_before = class_distribution(cids_raw)
df_after = class_distribution(cids)
df_compare = df_before.merge(
    df_after, on=['Class ID', 'Class Name'], how='outer', suffixes=(' (before)', ' (after)')
).fillna(0)
display(df_compare)

fig_pie = px.pie(
    df_after, values='Point Count', names='Class Name',
    title='Filtered 3D Point Cloud — Semantic Class Distribution',
    color='Class Name',
    color_discrete_map={CLASS_NAMES[c]: CLASS_HEX_COLORS[c] for c in df_after['Class ID']},
    hole=0.4,
)
fig_pie.update_layout(template='plotly_dark')
fig_pie.show()

Loaded 106,877 raw points from '../outputs/point_clouds/semantic_bridge_gpu.ply'
Camera-derived up vector: [ 0.111 -0.994  0.015]
Filtered -> 18,150 structure points (removed 88,727)
  removed by stage: {'background': 80216, 'statistical': 762, 'deck_plane': 1660, 'cable_envelope': 6089, 'cable_fan': 0}


,Class ID,Class Name,Point Count (before),Percentage (%) (before),Point Count (after),Percentage (%) (after)
0,0,background,80216,75.05,0.0,0.00
1,1,deck,10234,9.58,8178.0,45.06
2,2,stay_cable,9094,8.51,2754.0,15.17
3,3,tower,4464,4.18,4378.0,24.12
4,4,foundation,2869,2.68,2840.0,15.65


## 2. Interactive 3D — Filtered Structure

Rotate (left click), pan (right click), zoom (scroll). Toggle legend items to show/hide classes.

In [3]:
fig_3d = create_interactive_3d_figure(xyz, rgb, cids, point_size=2.5)
fig_3d.update_layout(title=f"Filtered Semantic Bridge ({len(xyz):,} points)")
fig_3d.show()

## 3. Per-Class Views

Background is already removed by the filter. Below: larger markers for structural components only.

In [4]:
print(f"Structure-only cloud: {len(xyz):,} points across {len(np.unique(cids))} classes")
fig_struct = create_interactive_3d_figure(xyz, rgb, cids, point_size=3.0)
fig_struct.update_layout(title="Bridge Structural Components (filtered)")
fig_struct.show()

Structure-only cloud: 18,150 points across 4 classes
